# Markov random fields (undirected models) — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def normalize(v):
    v=np.asarray(v,dtype=float); return v/v.sum()
def softmax(v):
    v=np.asarray(v,dtype=float); e=np.exp(v-v.max()); return e/e.sum()
def norm_pdf(x,mu,var):
    return np.exp(-0.5*(np.asarray(x)-mu)**2/var)/np.sqrt(2*np.pi*var)
def beta_pdf_grid(x,a,b):
    B=math.gamma(a)*math.gamma(b)/math.gamma(a+b)
    return (np.asarray(x)**(a-1))*((1-np.asarray(x))**(b-1))/B
print('setup ok')

## Undirected potentials score compatible assignments, then the partition function normalizes them
Markov random fields model mutual compatibility rather than parent-child causality. These five examples compute Ising-style potentials, the partition function, marginals, coupling strength, and a conditional independence pattern.

In [ ]:
# 1) two-spin Ising MRF: same labels receive higher potential when J>0
states=np.array(list(itertools.product([-1,1],repeat=2))); J=0.8; scores=np.exp(J*states[:,0]*states[:,1]); probs=scores/scores.sum()
plt.figure(figsize=(6,3)); plt.bar([str(tuple(s)) for s in states],probs); plt.title('aligned states are more probable')
assert abs(probs[0]-0.4160091925669622)<1e-12

In [ ]:
# 2) partition function Z normalizes unscaled scores
Z=scores.sum(); plt.figure(figsize=(6,3)); plt.bar(['Z','sum probs'],[Z,probs.sum()]); plt.title(f'Z={Z:.3f} makes probabilities sum to 1')
assert abs(Z-5.349739785219379)<1e-12 and abs(probs.sum()-1)<1e-12

In [ ]:
# 3) marginal of one spin remains balanced by symmetry
marg_plus=probs[states[:,0]==1].sum(); plt.figure(figsize=(6,3)); plt.bar(['X=-1','X=+1'],[1-marg_plus,marg_plus]); plt.title('coupling creates dependence, not marginal bias')
assert abs(marg_plus-0.5)<1e-12

In [ ]:
# 4) stronger coupling increases correlation E[X1 X2]
Js=np.linspace(0,2,80); corr=[]
for j in Js:
    sc=np.exp(j*states[:,0]*states[:,1]); pr=sc/sc.sum(); corr.append((pr*states[:,0]*states[:,1]).sum())
plt.figure(figsize=(6,3)); plt.plot(Js,corr); plt.xlabel('J'); plt.ylabel('correlation'); plt.title('edge strength controls dependence')
assert corr[-1]>corr[0]

In [ ]:
# 5) in a chain, endpoints become independent when the middle is fixed
chain=np.array(list(itertools.product([-1,1],repeat=3))); J=0.7; sc=np.exp(J*(chain[:,0]*chain[:,1]+chain[:,1]*chain[:,2])); pr=sc/sc.sum()
cond=pr[chain[:,1]==1]; xs=chain[chain[:,1]==1]; cond=cond/cond.sum(); joint_end=np.zeros((2,2))
for p,s in zip(cond,xs): joint_end[int(s[0]==1),int(s[2]==1)]+=p
prod=np.outer(joint_end.sum(1),joint_end.sum(0))
plt.figure(figsize=(6,3)); plt.imshow(joint_end); plt.colorbar(); plt.title('P(X1,X3 | X2=1) factorizes')
assert np.allclose(joint_end,prod)